# Wasserstein Ascend Descend
### (C. and N.) Garcia Trillos

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
import numpy as np
from Robust_nn.WAD import WAD2scale
import matplotlib.pyplot as plt
from utils.utils import read_vision_dataset
from utils.convnet import ConvNet
from utils.convnet_silu import ConvNetSiLU
import os
import torch
import gc
import torch.nn as nn
from torch.optim.swa_utils import AveragedModel, SWALR


**Read the DataLoader**

In [3]:
# Create some networks

# n_nets = 5
n_nets = 5
net_lst = [ConvNet() for i in range(n_nets)]
avg_nets =[ConvNet() for i in range(n_nets*4)]

# adv_net = WAD2scale(net_list = net_lst, avg_nets = avg_nets, 
#                     dataset_name='MNIST',batch_size = 1024, 
#                     device = None , criterion= nn.CrossEntropyLoss(), 
#                     scale_factor=5, num_adverse=2,
#                     kappa = { 'param': 0.2, 'adv': 0.2   },
#                     max_batches= 5)


adv_net = WAD2scale(net_list = net_lst, avg_nets = avg_nets, 
                    dataset_name='MNIST',batch_size = 64, 
                    device = None , criterion= nn.CrossEntropyLoss(), 
                    scale_factor=5, num_adverse=2,
                    penalty_coef = 0.1,
                    kappa = { 'param': 0.2, 'adv': 0.2   },
                    max_batches= 10)

In [4]:
import os
os.listdir('.')

['.git',
 '.vscode',
 'checkpoint',
 'Robust_nn',
 'Robust_reg_code.zip',
 'Test1_kappa0p2-4',
 'Tests_o1o2.ipynb',
 'Tests_WAD.ipynb',
 'Tests_WAD_script.py',
 'todo.md',
 'utils',
 '_avg_adv_folder']

In [5]:
adv_net.set_optimizer()
adv_net.train(epochs = 5)


Epoch: 0
tensor([0.5000, 0.5000])
tensor([0.2000, 0.2000, 0.2000, 0.2000, 0.2000])
0|1|2|3|4|5|6|7|8|9|10|
Epoch: 1
tensor([0.4725, 0.5275])
tensor([0.2006, 0.2005, 0.1994, 0.1988, 0.2007], grad_fn=<DivBackward0>)
0|1|2|3|4|5|6|7|8|9|10|
Epoch: 2
tensor([0.4626, 0.5374])
tensor([0.2007, 0.2021, 0.2002, 0.1980, 0.1990], grad_fn=<DivBackward0>)
0|1|2|3|4|5|6|7|8|9|10|
Epoch: 3
tensor([0.4528, 0.5472])
tensor([0.2004, 0.2038, 0.2027, 0.1987, 0.1944], grad_fn=<DivBackward0>)
0|1|2|3|4|5|6|7|8|9|10|
Epoch: 4
tensor([0.4433, 0.5567])
tensor([0.2009, 0.2065, 0.2073, 0.1990, 0.1863], grad_fn=<DivBackward0>)
0|1|2|3|4|5|6|7|8|9|10|

In [6]:
adv_net.save_model('Test1_kappa0p2-4')

Saving...
done.


In [7]:
adv_net.set_optimizer()
adv_net.import_model('Test1_kappa0p2-4')

Loading...
done.


In [8]:
'''
Testing the model 

Peform the following tests:
1. Test for accuracy of the average  model
2. Train directly the model with the whole set of adversaries. Calculate...
3. Create directly the adversarial. 
'''

'\nTesting the model \n\nPeform the following tests:\n1. Test for accuracy of the average  model\n2. Train directly the model with the whole set of adversaries. Calculate...\n3. Create directly the adversarial. \n'

In [9]:
adv_net.test_pgd(5)

epoch = 5, adv_acc = 30.73, clean_acc = 60.48
Saving the best model to ./checkpoint
Saving...
done.


(1.9188978611284, 30.73, 60.48)

In [10]:
adv_net.test_base()

0|1|2|3|4|5|6|7|8|9|10|
Loss of time-average model and adversaries tensor(-22.5353, grad_fn=<AddBackward0>)


tensor(-22.5353, grad_fn=<AddBackward0>)

In [11]:
adv_net.test_improve_adversaries(5)

0|1|2|3|4|5|6|7|8|9|10|
Loss of time-average after training model with available adversaries -22.482577776908876


-22.482577776908876

In [12]:
adv_net.test_improve_model(5)

0|1|2|3|4|5|6|7|8|9|10|
Loss of time-average after training model with available adversaries -22.441696763038635


-22.441696763038635